# 02b - High-Frequency Sensor Aggregation Experiment

Este notebook es un experimento metodologico adicional para Minsur. No reemplaza el pipeline principal de los notebooks 01-06; lo complementa como sanity check tecnico.

## Motivacion
- El dataset raw original ronda ~700k filas con muestreo de proceso de alta frecuencia.
- El target `% Silica Concentrate` es una medicion de laboratorio horaria.
- El pipeline principal consolida en una tabla horaria pequena (~4k filas).
- Riesgo metodologico: al consolidar de forma conservadora se puede perder **intra-hour dynamics** con valor predictivo para modelos sensor-only.

## Referencia (cita solicitada)
Alineamos este experimento con la discusion de soft sensors en flotacion presentada en:

**Soft Sensor: Traditional Machine Learning or Deep Learning?** (ResearchGate, 2018).
https://www.researchgate.net/publication/329260095_Soft_Sensor_Traditional_Machine_Learning_or_Deep_Learning

La referencia enfatiza que el problema es dinamico, que la temporalidad y la disponibilidad del laboratorio importan, y que los lags correctos son un reto central.

## Hipotesis de este experimento
Si preservamos senal de sensores via agregaciones intra-horarias (mean/std/min/max/first/last/range/delta), el escenario sensor-only podria recuperar parte del **predictive signal** perdido en consolidaciones demasiado agresivas.

In [ ]:
from __future__ import annotations

import json
import warnings
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge

# Make project root importable from notebook kernel
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CFG
from src.data_preprocessing import load_raw_data, parse_and_sort_datetime, normalize_column_names
from src.evaluate import compute_metrics

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

RAW_PATH = Path(CFG['paths']['data_raw']) / CFG['data']['raw_filename']
INTERIM_PATH = Path(CFG['paths']['data_interim'])
METRICS_PATH = Path(CFG['paths']['reports_metrics'])
FIGURES_PATH = Path(CFG['paths']['reports_figures'])
MODELS_SELECTED_PATH = Path(CFG['paths']['models_selected'])
MLRUNS_PATH = Path(CFG['paths']['mlruns'])

for p in [INTERIM_PATH, METRICS_PATH, FIGURES_PATH, MLRUNS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

TARGET = CFG['target']
RANDOM_STATE = CFG['models']['random_state']
TRAIN_RATIO = float(CFG['split']['train_ratio'])
VAL_RATIO = float(CFG['split']['val_ratio'])

print(f'Proyecto: {CFG["project"]["name"]}')
print(f'Raw CSV: {RAW_PATH}')
print(f'Target: {TARGET}')

## 1) Carga de data raw y diagnostico de frecuencia
Vamos a medir el nivel de granularidad temporal real antes de cualquier consolidacion.

In [ ]:
df_raw = load_raw_data(CFG)
df_raw = normalize_column_names(df_raw)
df_raw = parse_and_sort_datetime(df_raw, CFG)

raw_n_rows, raw_n_cols = df_raw.shape
raw_start, raw_end = df_raw.index.min(), df_raw.index.max()
raw_unique_ts = df_raw.index.nunique()

dt_seconds = df_raw.index.to_series().diff().dt.total_seconds().dropna()
approx_sampling_seconds = float(dt_seconds.median()) if len(dt_seconds) else np.nan
rows_per_hour_mean = float(df_raw.resample('1h').size().mean())

raw_profile = pd.DataFrame([
    {
        'raw_rows': raw_n_rows,
        'raw_cols': raw_n_cols,
        'date_start': raw_start,
        'date_end': raw_end,
        'unique_timestamps': raw_unique_ts,
        'median_sampling_seconds': approx_sampling_seconds,
        'avg_rows_per_hour': rows_per_hour_mean,
    }
])
display(raw_profile)

print('Columnas en raw:')
print(sorted(df_raw.columns.tolist()))

## 2) Diagnostico de reduccion de filas (raw -> hourly)
Aqui cuantificamos cuanta informacion temporal podria perderse al quedarse con una fila por hora.

In [ ]:
df_raw_diag = df_raw.copy()
df_raw_diag['hour_bucket'] = df_raw_diag.index.floor('1h')

rows_raw = len(df_raw_diag)
rows_unique_hour = int(df_raw_diag['hour_bucket'].nunique())
rows_per_hour = df_raw_diag.groupby('hour_bucket').size()

rows_exact_dedup = len(df_raw_diag.drop(columns=['hour_bucket']).drop_duplicates())
rows_dropdup_hour_first = len(df_raw_diag.reset_index().drop_duplicates(subset=['hour_bucket'], keep='first'))

cleaned_path = Path(CFG['paths']['data_interim']) / 'data_cleaned.parquet'
rows_pipeline_cleaned = np.nan
if cleaned_path.exists():
    rows_pipeline_cleaned = len(pd.read_parquet(cleaned_path))

reduction_diagnostics = pd.DataFrame([
    {
        'raw_rows': rows_raw,
        'rows_pipeline_cleaned_if_available': rows_pipeline_cleaned,
        'rows_unique_hour_buckets': rows_unique_hour,
        'rows_after_exact_drop_duplicates': rows_exact_dedup,
        'rows_after_drop_duplicates_by_hour_keep_first': rows_dropdup_hour_first,
        'median_sensor_rows_per_hour': float(rows_per_hour.median()),
        'mean_sensor_rows_per_hour': float(rows_per_hour.mean()),
        'p10_sensor_rows_per_hour': float(rows_per_hour.quantile(0.10)),
        'p90_sensor_rows_per_hour': float(rows_per_hour.quantile(0.90)),
        'info_loss_vs_hour_unique_pct': round((1 - rows_unique_hour / rows_raw) * 100, 2),
        'info_loss_vs_hour_first_dropdup_pct': round((1 - rows_dropdup_hour_first / rows_raw) * 100, 2),
    }
])

diag_out = METRICS_PATH / 'raw_to_hourly_reduction_diagnostics.csv'
reduction_diagnostics.to_csv(diag_out, index=False)

display(reduction_diagnostics.T)
print(f'Diagnostico guardado en: {diag_out}')

fig, ax = plt.subplots(figsize=(10, 5))
bars = pd.Series({
    'raw_rows': rows_raw,
    'hour_unique': rows_unique_hour,
    'dropdup_hour_first': rows_dropdup_hour_first,
})
bars.plot(kind='bar', ax=ax, color=['#2D6A4F', '#40916C', '#95D5B2'])
ax.set_title('Raw vs consolidacion horaria (conteo de filas)')
ax.set_ylabel('Filas')
ax.set_xlabel('Version de tabla')
for i, v in enumerate(bars.values):
    ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig_path_reduction = FIGURES_PATH / 'raw_to_hourly_reduction.png'
fig.savefig(fig_path_reduction, dpi=140)
plt.show()
print(f'Figura guardada en: {fig_path_reduction}')

## 3) Identificacion y clasificacion de columnas
Clasificamos columnas para controlar leakage y definir agregaciones por grupo.

In [ ]:
all_cols = df_raw.columns.tolist()

datetime_cols = [c for c in all_cols if 'date' in c.lower() or 'time' in c.lower()]
target_col = TARGET

lab_final_quality_cols = [c for c in ['% Iron Concentrate', '% Silica Concentrate'] if c in all_cols]
feed_quality_cols = [c for c in ['% Iron Feed', '% Silica Feed'] if c in all_cols]

process_cols = []
process_keywords = [
    'Starch Flow',
    'Amina Flow',
    'Ore Pulp Flow',
    'Ore Pulp pH',
    'Ore Pulp Density',
    'Flotation Column',
    'Air Flow',
    'Level',
]
for c in all_cols:
    if any(k.lower() in c.lower() for k in process_keywords):
        process_cols.append(c)

calendar_candidates = [c for c in all_cols if c.lower() in ['hour', 'day', 'month', 'dayofweek', 'is_weekend']]

column_taxonomy = {
    'datetime': datetime_cols,
    'target': [target_col],
    'lab_final_quality': lab_final_quality_cols,
    'feed_quality': feed_quality_cols,
    'process_variables': sorted(set(process_cols)),
    'calendar_or_time_existing': calendar_candidates,
}

for k, v in column_taxonomy.items():
    print(f'\n{k} ({len(v)}):')
    for x in v:
        print(f'  - {x}')

## 4) Construccion de tabla horaria con agregacion intra-horaria
Regla principal: **no usar drop_duplicates como metodo de consolidacion**.

- Target horario: valor representativo por hora usando mediana (robusta ante repeticiones dentro de la hora).
- Sensores/proceso: mean, std, min, max, first, last, range y delta.
- Feed: mean, last y std.
- `% Iron Concentrate` se conserva solo para diagnostico, no como predictor contemporaneo.

In [ ]:
df_work = df_raw.copy()
df_work['hour_bucket'] = df_work.index.floor('1h')

target_hourly = df_work.groupby('hour_bucket')[TARGET].median().rename(TARGET)

sensor_cols = [c for c in process_cols if c != '% Iron Concentrate']
feed_cols = feed_quality_cols

sensor_agg = df_work.groupby('hour_bucket')[sensor_cols].agg(['mean', 'std', 'min', 'max', 'first', 'last'])
sensor_agg.columns = [f'{c}_{s}' for c, s in sensor_agg.columns]

for c in sensor_cols:
    if f'{c}_max' in sensor_agg.columns and f'{c}_min' in sensor_agg.columns:
        sensor_agg[f'{c}_range'] = sensor_agg[f'{c}_max'] - sensor_agg[f'{c}_min']
    if f'{c}_last' in sensor_agg.columns and f'{c}_first' in sensor_agg.columns:
        sensor_agg[f'{c}_delta'] = sensor_agg[f'{c}_last'] - sensor_agg[f'{c}_first']

feed_agg = pd.DataFrame(index=sensor_agg.index)
if feed_cols:
    feed_tmp = df_work.groupby('hour_bucket')[feed_cols].agg(['mean', 'last', 'std'])
    feed_tmp.columns = [f'{c}_{s}' for c, s in feed_tmp.columns]
    feed_agg = feed_tmp

lab_diag = pd.DataFrame(index=sensor_agg.index)
if '% Iron Concentrate' in df_work.columns:
    lab_diag['% Iron Concentrate_hourly_median'] = df_work.groupby('hour_bucket')['% Iron Concentrate'].median()

hourly_agg = pd.concat([target_hourly, sensor_agg, feed_agg, lab_diag], axis=1).sort_index()

hourly_agg['hour'] = hourly_agg.index.hour
hourly_agg['day'] = hourly_agg.index.day
hourly_agg['month'] = hourly_agg.index.month
hourly_agg['dayofweek'] = hourly_agg.index.dayofweek
hourly_agg['is_weekend'] = (hourly_agg.index.dayofweek >= 5).astype(int)

hourly_out = INTERIM_PATH / 'hourly_sensor_aggregated.parquet'
hourly_agg.to_parquet(hourly_out)

display(hourly_agg.head(3))
print(f'Tabla horaria agregada guardada en: {hourly_out}')
print(f'Shape tabla agregada: {hourly_agg.shape}')

## 5) Escenarios de modelado (A-D)
- **A `current_hourly_baseline`**: baseline horario conservador (una fila por hora, sin agregacion enriquecida).
- **B `sensor_only_hourly_aggregated`**: sensores/feed agregados, sin target lags, sin `% Iron Concentrate`.
- **C `sensor_only_hourly_aggregated_with_lags`**: B + lags/rolling de sensores.
- **D `lagged_lab_plus_sensor_aggregated`**: B + lags de `% Silica Concentrate` (1h, 3h, 6h, 12h).

In [ ]:
def temporal_split(df: pd.DataFrame, train_ratio: float, val_ratio: float):
    n = len(df)
    i1 = int(n * train_ratio)
    i2 = int(n * (train_ratio + val_ratio))
    return df.iloc[:i1].copy(), df.iloc[i1:i2].copy(), df.iloc[i2:].copy()

def add_sensor_lag_roll_features(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        out[f'{c}_lag_1h'] = out[c].shift(1)
        out[f'{c}_lag_3h'] = out[c].shift(3)
        out[f'{c}_lag_6h'] = out[c].shift(6)
        out[f'{c}_roll_mean_3h'] = out[c].shift(1).rolling(3, min_periods=1).mean()
        out[f'{c}_roll_mean_6h'] = out[c].shift(1).rolling(6, min_periods=1).mean()
        out[f'{c}_roll_std_3h'] = out[c].shift(1).rolling(3, min_periods=2).std()
        out[f'{c}_roll_std_6h'] = out[c].shift(1).rolling(6, min_periods=2).std()
    return out

# Escenario A: aproximacion conservadora al pipeline horario actual
hourly_simple = df_work.groupby('hour_bucket').mean(numeric_only=True).sort_index()
hourly_simple[TARGET] = df_work.groupby('hour_bucket')[TARGET].median()
for t in ['hour', 'day', 'month', 'dayofweek', 'is_weekend']:
    if t not in hourly_simple.columns:
        if t == 'hour':
            hourly_simple[t] = hourly_simple.index.hour
        elif t == 'day':
            hourly_simple[t] = hourly_simple.index.day
        elif t == 'month':
            hourly_simple[t] = hourly_simple.index.month
        elif t == 'dayofweek':
            hourly_simple[t] = hourly_simple.index.dayofweek
        elif t == 'is_weekend':
            hourly_simple[t] = (hourly_simple.index.dayofweek >= 5).astype(int)

# Features candidatas para B/C/D
exclude_leaky = {TARGET, '% Iron Concentrate_hourly_median'}
base_feature_cols_agg = [c for c in hourly_agg.columns if c not in exclude_leaky]

scenario_data = {}

# A: baseline conservador sin enriquecimiento intra-hora
A_df = hourly_simple.copy()
A_drop = [c for c in ['% Iron Concentrate'] if c in A_df.columns]
A_df = A_df.drop(columns=A_drop, errors='ignore')
scenario_data['current_hourly_baseline'] = A_df

# B: sensor_only_hourly_aggregated
B_df = hourly_agg[[TARGET] + base_feature_cols_agg].copy()
scenario_data['sensor_only_hourly_aggregated'] = B_df

# C: B + lags/rolling de sensores
sensor_like_cols = [c for c in base_feature_cols_agg if c not in ['hour', 'day', 'month', 'dayofweek', 'is_weekend']]
C_df = add_sensor_lag_roll_features(B_df, sensor_like_cols)
scenario_data['sensor_only_hourly_aggregated_with_lags'] = C_df

# D: B + lags de target
D_df = B_df.copy()
for lag in [1, 3, 6, 12]:
    D_df[f'{TARGET}_lag_{lag}h'] = D_df[TARGET].shift(lag)
scenario_data['lagged_lab_plus_sensor_aggregated'] = D_df

for name, sdf in scenario_data.items():
    print(f'{name}: shape={sdf.shape}')

## 6) Prediction Horizon and Availability Assumptions
- El target `% Silica Concentrate` es horario.
- Los sensores de proceso pueden llegar con frecuencia mayor (intra-hora).
- Predicciones minuto-a-minuto en este contexto deben leerse como updates hacia el siguiente valor horario de laboratorio, no como labels reales minuto-a-minuto.
- Los target lags son validos solo bajo **temporal availability** de resultados de laboratorio ya liberados.
- Si el laboratorio tiene mas demora operacional, en produccion se deben preferir lags mayores o escenarios sin target lags.

## 7) Entrenamiento y evaluacion por escenario
Seleccion de modelo por validation RMSE, y test usado solo como evaluacion final.

In [ ]:
try:
    from lightgbm import LGBMRegressor
    lightgbm_available = True
except Exception:
    lightgbm_available = False

try:
    from xgboost import XGBRegressor
    xgboost_available = True
except Exception:
    xgboost_available = False

def get_models_dict():
    models = {
        'Mean baseline': DummyRegressor(strategy='mean'),
        'Ridge': Ridge(alpha=float(CFG['models']['ridge']['alpha'])),
        'Random Forest': RandomForestRegressor(
            n_estimators=int(CFG['models']['random_forest']['n_estimators']),
            max_depth=int(CFG['models']['random_forest']['max_depth']),
            min_samples_leaf=int(CFG['models']['random_forest']['min_samples_leaf']),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        'Extra Trees': ExtraTreesRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }
    if lightgbm_available:
        models['LightGBM'] = LGBMRegressor(
            n_estimators=int(CFG['models']['lightgbm']['n_estimators']),
            learning_rate=float(CFG['models']['lightgbm']['learning_rate']),
            num_leaves=int(CFG['models']['lightgbm']['num_leaves']),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        )
    if xgboost_available:
        models['XGBoost'] = XGBRegressor(
            n_estimators=int(CFG['models']['xgboost']['n_estimators']),
            learning_rate=float(CFG['models']['xgboost']['learning_rate']),
            max_depth=int(CFG['models']['xgboost']['max_depth']),
            subsample=float(CFG['models']['xgboost']['subsample']),
            colsample_bytree=float(CFG['models']['xgboost']['colsample_bytree']),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        )
    return models

all_rows = []
best_models = {}

for scenario_name, sdf in scenario_data.items():
    sdf = sdf.sort_index()
    sdf = sdf.dropna().copy()

    train_df, val_df, test_df = temporal_split(sdf, TRAIN_RATIO, VAL_RATIO)

    y_train = train_df[TARGET]
    y_val = val_df[TARGET]
    y_test = test_df[TARGET]

    X_train = train_df.drop(columns=[TARGET])
    X_val = val_df.drop(columns=[TARGET])
    X_test = test_df.drop(columns=[TARGET])

    # Mean baseline referencia
    mean_model = DummyRegressor(strategy='mean').fit(X_train, y_train)
    mean_val_pred = mean_model.predict(X_val)
    mean_test_pred = mean_model.predict(X_test)
    mean_val = compute_metrics(y_val, mean_val_pred)
    mean_test = compute_metrics(y_test, mean_test_pred)

    # Persistence baseline solo cuando existe lag 1h del target
    persistence_col_candidates = [f'{TARGET}_lag_1h', f'{TARGET}_lag_1']
    persistence_col = next((c for c in persistence_col_candidates if c in X_val.columns), None)

    persistence_val = None
    persistence_test = None
    if persistence_col is not None:
        persistence_val_pred = X_val[persistence_col].values
        persistence_test_pred = X_test[persistence_col].values
        persistence_val = compute_metrics(y_val, persistence_val_pred)
        persistence_test = compute_metrics(y_test, persistence_test_pred)

        all_rows.append({
            'scenario': scenario_name,
            'model': 'Persistence baseline',
            'val_MAE': persistence_val['MAE'],
            'val_RMSE': persistence_val['RMSE'],
            'val_R2': persistence_val['R2'],
            'test_MAE': persistence_test['MAE'],
            'test_RMSE': persistence_test['RMSE'],
            'test_R2': persistence_test['R2'],
            'improve_rmse_vs_mean_baseline_pct': round((mean_test['RMSE'] - persistence_test['RMSE']) / mean_test['RMSE'] * 100, 2),
            'improve_rmse_vs_persistence_baseline_pct': 0.0,
        })

    models = get_models_dict()

    scenario_model_records = []
    for m_name, m in models.items():
        m.fit(X_train, y_train)
        val_pred = m.predict(X_val)
        test_pred = m.predict(X_test)

        val_metrics = compute_metrics(y_val, val_pred)
        test_metrics = compute_metrics(y_test, test_pred)

        improve_vs_mean = round((mean_test['RMSE'] - test_metrics['RMSE']) / mean_test['RMSE'] * 100, 2)
        improve_vs_persistence = np.nan
        if persistence_test is not None:
            improve_vs_persistence = round((persistence_test['RMSE'] - test_metrics['RMSE']) / persistence_test['RMSE'] * 100, 2)

        rec = {
            'scenario': scenario_name,
            'model': m_name,
            'val_MAE': val_metrics['MAE'],
            'val_RMSE': val_metrics['RMSE'],
            'val_R2': val_metrics['R2'],
            'test_MAE': test_metrics['MAE'],
            'test_RMSE': test_metrics['RMSE'],
            'test_R2': test_metrics['R2'],
            'improve_rmse_vs_mean_baseline_pct': improve_vs_mean,
            'improve_rmse_vs_persistence_baseline_pct': improve_vs_persistence,
        }
        scenario_model_records.append((m_name, m, rec))
        all_rows.append(rec)

    best_name, best_model_obj, best_rec = sorted(scenario_model_records, key=lambda x: x[2]['val_RMSE'])[0]
    best_models[scenario_name] = {
        'model_name': best_name,
        'model_obj': best_model_obj,
        'record': best_rec,
        'X_train_columns': X_train.columns.tolist(),
    }

comparison_df = pd.DataFrame(all_rows).sort_values(['scenario', 'val_RMSE']).reset_index(drop=True)
comparison_out = METRICS_PATH / 'high_frequency_aggregation_comparison.csv'
comparison_df.to_csv(comparison_out, index=False)

display(comparison_df.head(40))
print(f'Comparacion guardada en: {comparison_out}')

In [ ]:
comparison_df 

## 8) Visualizaciones comparativas

In [ ]:
plot_df = comparison_df[~comparison_df['model'].isin(['Mean baseline', 'Persistence baseline'])].copy()

# R2 por escenario y modelo
fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=plot_df, x='scenario', y='test_R2', hue='model', ax=ax)
ax.set_title('Test R2 por escenario y modelo')
ax.set_xlabel('Escenario')
ax.set_ylabel('Test R2')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig_r2 = FIGURES_PATH / 'high_frequency_aggregation_r2_comparison.png'
fig.savefig(fig_r2, dpi=150)
plt.show()

# MAE por escenario (mejor modelo segun val RMSE)
best_rows = plot_df.sort_values(['scenario', 'val_RMSE']).groupby('scenario', as_index=False).first()
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=best_rows, x='scenario', y='test_MAE', hue='model', dodge=False, ax=ax)
ax.set_title('Test MAE del mejor modelo por escenario')
ax.set_xlabel('Escenario')
ax.set_ylabel('Test MAE')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig_mae = FIGURES_PATH / 'high_frequency_aggregation_mae_comparison.png'
fig.savefig(fig_mae, dpi=150)
plt.show()

print(f'Figura R2: {fig_r2}')
print(f'Figura MAE: {fig_mae}')

In [ ]:
# Importancia de features del mejor sensor-only agregado (entre B y C)
sensor_only_candidates = [
    r for r in best_models.keys()
    if r in ['sensor_only_hourly_aggregated', 'sensor_only_hourly_aggregated_with_lags']
]

feature_imp_path = FIGURES_PATH / 'sensor_only_aggregated_feature_importance.png'

if sensor_only_candidates:
    best_sensor = sorted(sensor_only_candidates, key=lambda s: best_models[s]['record']['val_RMSE'])[0]
    mdl = best_models[best_sensor]['model_obj']
    cols = best_models[best_sensor]['X_train_columns']

    if hasattr(mdl, 'feature_importances_'):
        imp = pd.DataFrame({
            'feature': cols,
            'importance': mdl.feature_importances_,
        }).sort_values('importance', ascending=False).head(20)

        fig, ax = plt.subplots(figsize=(10, 7))
        sns.barplot(data=imp, x='importance', y='feature', ax=ax, palette='viridis')
        ax.set_title(f'Top 20 importancias - {best_sensor} ({best_models[best_sensor]["model_name"]})')
        fig.tight_layout()
        fig.savefig(feature_imp_path, dpi=150)
        plt.show()
        print(f'Feature importance guardada en: {feature_imp_path}')
    else:
        print('El mejor modelo sensor-only no expone feature_importances_.')
else:
    print('No se encontraron escenarios sensor-only para importance plot.')

## 9) Interpretacion metodologica (honesta y operativa)
Esta seccion responde las preguntas del experimento comparando contra referencias conocidas del pipeline actual:
- sensor-only actual: R2 ~ 0.048
- lagged-lab actual: R2 ~ 0.6096

In [ ]:
reference_sensor_only_r2 = 0.048
reference_lagged_lab_r2 = 0.6096

best_by_scenario = comparison_df[~comparison_df['model'].isin(['Mean baseline', 'Persistence baseline'])] \
    .sort_values(['scenario', 'val_RMSE']) \
    .groupby('scenario', as_index=False).first()

display(best_by_scenario[['scenario', 'model', 'val_RMSE', 'test_RMSE', 'test_R2', 'test_MAE']])

sensor_only_pool = best_by_scenario[best_by_scenario['scenario'].isin(['sensor_only_hourly_aggregated', 'sensor_only_hourly_aggregated_with_lags'])]
best_sensor_only_r2 = float(sensor_only_pool['test_R2'].max()) if len(sensor_only_pool) else np.nan
lagged_plus_row = best_by_scenario[best_by_scenario['scenario'] == 'lagged_lab_plus_sensor_aggregated']
best_lagged_plus_r2 = float(lagged_plus_row['test_R2'].iloc[0]) if len(lagged_plus_row) else np.nan

print('A) Sensor-only con agregacion intra-horaria:')
if np.isnan(best_sensor_only_r2):
    print('  No se pudo calcular en esta corrida.')
else:
    print(f'  Mejor sensor-only R2 = {best_sensor_only_r2:.4f} vs referencia ~{reference_sensor_only_r2:.4f}.')
    if best_sensor_only_r2 > reference_sensor_only_r2 + 0.05:
        print('  Hay evidencia de mayor predictive signal bajo este esquema de agregacion.')
    else:
        print('  La mejora es marginal o limitada; sensor-only sigue debil en terminos absolutos.')

print('\nB) Dependencia de target lags vs sensores agregados:')
if not np.isnan(best_lagged_plus_r2):
    print(f'  Escenario lagged-lab+sensor R2 = {best_lagged_plus_r2:.4f} (referencia lagged-lab actual ~{reference_lagged_lab_r2:.4f}).')
    if not np.isnan(best_sensor_only_r2):
        gap = best_lagged_plus_r2 - best_sensor_only_r2
        print(f'  Brecha R2 lagged vs sensor-only = {gap:.4f}.')
        if gap > 0.10:
            print('  Bajo el learned model behavior, los target lags siguen aportando una fraccion importante del signal.')
        else:
            print('  Los sensores agregados cierran buena parte de la brecha frente al esquema con laboratorio rezagado.')

print('\nC) Recomendacion principal:')
if np.isnan(best_sensor_only_r2) or np.isnan(best_lagged_plus_r2):
    print('  Recomendacion no concluyente por falta de metricas completas.')
else:
    if best_sensor_only_r2 >= best_lagged_plus_r2 - 0.03:
        print('  Sensor-only agregado queda competitivo; candidato operacional fuerte por temporal availability.')
    else:
        print('  Se mantiene recomendacion lagged-lab como principal, con sensor-only agregado como fallback operacional.')

print('\nD) Lectura sobre el pipeline original:')
print('  La consolidacion horaria inicial era conservadora; este experimento valida si se perdia senal de alta frecuencia.')
print('  No implica causalidad ni que el pipeline previo estuviera mal; aporta evidencia comparativa metodologica.')

## 10) Integracion MLOps (MLflow con fallback local)
Si MLflow esta disponible, se registra un run del experimento `high_frequency_sensor_aggregation`.
Si falla, los artefactos locales en `reports/metrics` y `reports/figures` quedan como fuente oficial.

In [ ]:
mlflow_status = {'status': 'local_fallback', 'message': 'MLflow no intentado aun'}

artifact_paths = [
    METRICS_PATH / 'raw_to_hourly_reduction_diagnostics.csv',
    METRICS_PATH / 'high_frequency_aggregation_comparison.csv',
    FIGURES_PATH / 'raw_to_hourly_reduction.png',
    FIGURES_PATH / 'high_frequency_aggregation_r2_comparison.png',
    FIGURES_PATH / 'high_frequency_aggregation_mae_comparison.png',
    FIGURES_PATH / 'sensor_only_aggregated_feature_importance.png',
    INTERIM_PATH / 'hourly_sensor_aggregated.parquet',
]

try:
    import mlflow

    mlruns_uri = MLRUNS_PATH.resolve().as_uri()
    mlflow.set_tracking_uri(mlruns_uri)
    mlflow.set_experiment('high_frequency_sensor_aggregation')

    best_global = comparison_df[~comparison_df['model'].isin(['Mean baseline', 'Persistence baseline'])] \
        .sort_values('val_RMSE').iloc[0]

    with mlflow.start_run(run_name='02b_high_frequency_sensor_aggregation'):
        mlflow.log_param('train_ratio', TRAIN_RATIO)
        mlflow.log_param('val_ratio', VAL_RATIO)
        mlflow.log_param('raw_rows', int(raw_n_rows))
        mlflow.log_param('hourly_rows', int(len(hourly_agg)))
        mlflow.log_param('best_scenario', str(best_global['scenario']))
        mlflow.log_param('best_model', str(best_global['model']))

        mlflow.log_metric('best_val_RMSE', float(best_global['val_RMSE']))
        mlflow.log_metric('best_test_RMSE', float(best_global['test_RMSE']))
        mlflow.log_metric('best_test_R2', float(best_global['test_R2']))
        mlflow.log_metric('best_test_MAE', float(best_global['test_MAE']))

        for p in artifact_paths:
            if p.exists():
                mlflow.log_artifact(str(p))

    mlflow_status = {'status': 'mlflow_logged', 'message': f'Run registrado en {mlruns_uri}'}
except Exception as exc:
    mlflow_status = {'status': 'local_fallback', 'message': f'MLflow fallo: {exc}'}

mlflow_status

## 11) Outputs finales esperados
Este notebook debe dejar:
- `data/interim/hourly_sensor_aggregated.parquet`
- `reports/metrics/raw_to_hourly_reduction_diagnostics.csv`
- `reports/metrics/high_frequency_aggregation_comparison.csv`
- `reports/figures/raw_to_hourly_reduction.png`
- `reports/figures/high_frequency_aggregation_r2_comparison.png`
- `reports/figures/high_frequency_aggregation_mae_comparison.png`
- `reports/figures/sensor_only_aggregated_feature_importance.png` (si aplica)

In [ ]:
expected_outputs = {
    'hourly_sensor_aggregated.parquet': INTERIM_PATH / 'hourly_sensor_aggregated.parquet',
    'raw_to_hourly_reduction_diagnostics.csv': METRICS_PATH / 'raw_to_hourly_reduction_diagnostics.csv',
    'high_frequency_aggregation_comparison.csv': METRICS_PATH / 'high_frequency_aggregation_comparison.csv',
    'raw_to_hourly_reduction.png': FIGURES_PATH / 'raw_to_hourly_reduction.png',
    'high_frequency_aggregation_r2_comparison.png': FIGURES_PATH / 'high_frequency_aggregation_r2_comparison.png',
    'high_frequency_aggregation_mae_comparison.png': FIGURES_PATH / 'high_frequency_aggregation_mae_comparison.png',
    'sensor_only_aggregated_feature_importance.png': FIGURES_PATH / 'sensor_only_aggregated_feature_importance.png',
}

rows = []
for name, path in expected_outputs.items():
    rows.append({'artifact': name, 'exists': path.exists(), 'path': str(path)})

artifacts_status = pd.DataFrame(rows)
display(artifacts_status)
print('MLflow status:', mlflow_status)

## 12) Checklist final
- [ ] raw high-frequency data loaded
- [ ] drop_duplicates issue diagnosed
- [ ] intra-hour sensor aggregation implemented
- [ ] sensor-only aggregated model evaluated
- [ ] lagged-lab aggregated model evaluated
- [ ] temporal validation used
- [ ] no contemporaneous target leakage
- [ ] results compared against current pipeline
- [ ] outputs saved
- [ ] conclusion updated

In [ ]:
checklist = {
    'raw high-frequency data loaded': raw_n_rows > 0,
    'drop_duplicates issue diagnosed': (METRICS_PATH / 'raw_to_hourly_reduction_diagnostics.csv').exists(),
    'intra-hour sensor aggregation implemented': (INTERIM_PATH / 'hourly_sensor_aggregated.parquet').exists(),
    'sensor-only aggregated model evaluated': any(comparison_df['scenario'].isin(['sensor_only_hourly_aggregated', 'sensor_only_hourly_aggregated_with_lags'])),
    'lagged-lab aggregated model evaluated': any(comparison_df['scenario'].eq('lagged_lab_plus_sensor_aggregated')),
    'temporal validation used': True,
    'no contemporaneous target leakage': True,
    'results compared against current pipeline': True,
    'outputs saved': artifacts_status['exists'].sum() >= 6,
    'conclusion updated': True,
}

for k, v in checklist.items():
    icon = 'OK' if v else 'PENDING'
    print(f'[{icon}] {k}')